# Acoustic feature pilots, QC, and batches 0001-0400

These are original, unmodified Python source cells from the exported Kaggle notebook, organized by research phase. Execution outputs and counts have been cleared. Cells are in original index order within this phase. The Kaggle project workspace and some earlier phases are required; this notebook is not guaranteed to execute by itself from a clean environment. The original index for every cell is stored in cell metadata.

The historical record includes exploratory retries and some superseded checks. Do not rerun an early output-producing cell over frozen results without backing up the project first.

In [ ]:

import io
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa

from pathlib import Path

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
OUT = PROJECT / "features/acoustic/acoustic_smoke_test_5.csv"

OUT.parent.mkdir(parents=True, exist_ok=True)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

# Load first 400 metadata rows, preserving Parquet order
pf = pq.ParquetFile(TRAIN)

target_ids = set(pilot["audio_id"].head(5).tolist())

records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):

    table = pf.read_row_group(
        group_id,
        columns=["audio"]
    )

    for audio in table.column("audio").to_pylist():

        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        assert sr == 16000
        assert y.ndim == 1

        duration = len(y) / sr

        # Energy-based speech interval detection
        intervals = librosa.effects.split(
            y,
            top_db=30,
            frame_length=512,
            hop_length=160
        )

        speech_duration = sum(
            (end - start) / sr
            for start, end in intervals
        )

        # Internal pauses between speech intervals
        gaps = [
            (intervals[i + 1][0] - intervals[i][1]) / sr
            for i in range(len(intervals) - 1)
        ]

        pauses = [gap for gap in gaps if gap >= 0.2]
        pause_duration = sum(pauses)

        # YIN pitch: preliminary, unvoiced frames not filtered
        f0 = librosa.yin(
            y,
            fmin=75,
            fmax=500,
            sr=sr,
            frame_length=1024,
            hop_length=256
        )

        rms = librosa.feature.rms(
            y=y,
            frame_length=1024,
            hop_length=256
        )[0]

        records.append({
            "audio_id": audio_id,
            "duration_sec": round(duration, 4),
            "speech_duration_sec": round(speech_duration, 4),
            "pause_count": len(pauses),
            "pause_duration_sec": round(pause_duration, 4),
            "pause_ratio": round(pause_duration / duration, 4),
            "pitch_mean_hz_preliminary": round(float(np.mean(f0)), 2),
            "pitch_sd_hz_preliminary": round(float(np.std(f0)), 2),
            "rms_mean": round(float(np.mean(rms)), 5)
        })

    if found == target_ids:
        break

assert found == target_ids, f"Missing audio IDs: {target_ids - found}"

result = pd.DataFrame(records)

result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)

result = result.sort_values("pilot_index")

result.to_csv(OUT, index=False)

print("=== ACOUSTIC SMOKE TEST ===")
print(result.to_string(index=False))

print("\n=== VALIDATION ===")
print("Processed:", len(result))
print("Missing values:", result.isna().sum().sum())
print("Negative speech durations:",
      (result["speech_duration_sec"] < 0).sum())
print("Speech longer than audio:",
      (result["speech_duration_sec"] > result["duration_sec"]).sum())

print("\nSaved:", OUT)


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

OUTPUT = PROJECT / "features/acoustic/pitch_validation_5.csv"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string"}
)

target_ids = set(pilot["audio_id"].head(5))

pf = pq.ParquetFile(TRAIN)

results = []
found = set()

for group_id in range(pf.metadata.num_row_groups):

    table = pf.read_row_group(
        group_id,
        columns=["audio"]
    )

    for audio in table.column("audio").to_pylist():

        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        # Probabilistic YIN with voiced-frame detection
        f0, voiced_flag, voiced_prob = librosa.pyin(
            y,
            sr=sr,
            fmin=75,
            fmax=500,
            frame_length=1024,
            hop_length=256,
            fill_na=np.nan
        )

        voiced_f0 = f0[
            voiced_flag & np.isfinite(f0)
        ]

        results.append({
            "audio_id": audio_id,
            "total_frames": len(f0),
            "voiced_frames": len(voiced_f0),
            "voiced_ratio": round(
                len(voiced_f0) / len(f0), 4
            ),
            "pitch_mean_hz": (
                round(float(np.mean(voiced_f0)), 2)
                if len(voiced_f0) else np.nan
            ),
            "pitch_sd_hz": (
                round(float(np.std(voiced_f0)), 2)
                if len(voiced_f0) else np.nan
            ),
            "pitch_min_hz": (
                round(float(np.min(voiced_f0)), 2)
                if len(voiced_f0) else np.nan
            ),
            "pitch_max_hz": (
                round(float(np.max(voiced_f0)), 2)
                if len(voiced_f0) else np.nan
            )
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(results)

result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)

result = result.sort_values("pilot_index")

result.to_csv(OUTPUT, index=False)

print("=== VOICED PITCH VALIDATION ===")
print(result.to_string(index=False))

print("\n=== QUALITY CHECK ===")
print("Processed:", len(result))
print("No voiced frames:",
      (result["voiced_frames"] == 0).sum())
print("Missing pitch means:",
      result["pitch_mean_hz"].isna().sum())

print("\nSaved:", OUTPUT)


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

OUT = PROJECT / "features/acoustic/pause_diagnostics_5.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string"}
)

target_ids = set(pilot["audio_id"].head(5))

pf = pq.ParquetFile(TRAIN)
records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):

    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:

        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        duration = len(y) / sr

        # Energy-based speech segmentation
        intervals = librosa.effects.split(
            y,
            top_db=30,
            frame_length=512,
            hop_length=160
        )

        speech_sec = sum(
            (end - start) / sr for start, end in intervals
        )

        if len(intervals):
            leading_silence = intervals[0][0] / sr
            trailing_silence = (
                len(y) - intervals[-1][1]
            ) / sr
        else:
            leading_silence = duration
            trailing_silence = 0.0

        internal_gaps = [
            (intervals[i + 1][0] - intervals[i][1]) / sr
            for i in range(len(intervals) - 1)
        ]

        pauses_100 = [
            gap for gap in internal_gaps if gap >= 0.1
        ]
        pauses_200 = [
            gap for gap in internal_gaps if gap >= 0.2
        ]
        pauses_300 = [
            gap for gap in internal_gaps if gap >= 0.3
        ]

        records.append({
            "audio_id": audio_id,
            "duration_sec": round(duration, 3),
            "speech_sec": round(speech_sec, 3),
            "speech_segments": len(intervals),
            "leading_silence_sec": round(
                leading_silence, 3
            ),
            "trailing_silence_sec": round(
                trailing_silence, 3
            ),
            "internal_gaps": len(internal_gaps),
            "pause_count_100ms": len(pauses_100),
            "pause_count_200ms": len(pauses_200),
            "pause_count_300ms": len(pauses_300),
            "pause_duration_200ms": round(
                sum(pauses_200), 3
            ),
            "all_internal_gap_sec": round(
                sum(internal_gaps), 3
            )
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(records)
result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)
result = result.sort_values("pilot_index")

result.to_csv(OUT, index=False)

print("=== PAUSE DIAGNOSTICS ===")
print(result.to_string(index=False))

print("\n=== SUMMARY ===")
print("Processed:", len(result))
print("Zero speech segments:",
      (result["speech_segments"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\nSaved:", OUT)


In [ ]:

import io
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

OUT = PROJECT / "features/acoustic/silero_vad_diagnostics_5.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

# Load Silero VAD from torch hub
model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

get_speech_timestamps = utils[0]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string"}
)

target_ids = set(pilot["audio_id"].head(5))

pf = pq.ParquetFile(TRAIN)
records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):

    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        assert sr == 16000
        assert y.ndim == 1

        timestamps = get_speech_timestamps(
            torch.from_numpy(y),
            model,
            sampling_rate=sr,
            threshold=0.5,
            min_speech_duration_ms=100,
            min_silence_duration_ms=200,
            speech_pad_ms=0
        )

        duration = len(y) / sr

        speech_sec = sum(
            (segment["end"] - segment["start"]) / sr
            for segment in timestamps
        )

        gaps = [
            (
                timestamps[i + 1]["start"]
                - timestamps[i]["end"]
            ) / sr
            for i in range(len(timestamps) - 1)
        ]

        pauses = [gap for gap in gaps if gap >= 0.2]

        leading = (
            timestamps[0]["start"] / sr
            if timestamps else duration
        )
        trailing = (
            (len(y) - timestamps[-1]["end"]) / sr
            if timestamps else 0.0
        )

        records.append({
            "audio_id": audio_id,
            "duration_sec": round(duration, 3),
            "speech_segments": len(timestamps),
            "speech_sec": round(speech_sec, 3),
            "leading_silence_sec": round(leading, 3),
            "trailing_silence_sec": round(trailing, 3),
            "pause_count_200ms": len(pauses),
            "pause_duration_200ms": round(sum(pauses), 3),
            "speech_ratio": round(
                speech_sec / duration, 4
            )
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(records)

result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)

result = result.sort_values("pilot_index")
result.to_csv(OUT, index=False)

print("=== SILERO VAD DIAGNOSTICS ===")
print(result.to_string(index=False))

print("\n=== SUMMARY ===")
print("Processed:", len(result))
print("No speech detected:",
      (result["speech_segments"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\nSaved:", OUT)


In [ ]:

import io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
OUTPUT = PROJECT / "results/figures/segmentation_comparison_000700008.png"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

TARGET_ID = "000700008"
pf = pq.ParquetFile(TRAIN)

audio_bytes = None

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        if Path(audio["path"]).stem == TARGET_ID:
            audio_bytes = audio["bytes"]
            break

    if audio_bytes is not None:
        break

assert audio_bytes is not None, "Target audio not found"

y, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32")
duration = len(y) / sr

energy_intervals = librosa.effects.split(
    y,
    top_db=30,
    frame_length=512,
    hop_length=160
)

# Use the already-loaded Silero model if present.
if "model" not in globals() or "get_speech_timestamps" not in globals():
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]

vad_intervals = get_speech_timestamps(
    torch.from_numpy(y),
    model,
    sampling_rate=sr,
    threshold=0.5,
    min_speech_duration_ms=100,
    min_silence_duration_ms=200,
    speech_pad_ms=0
)

rms = librosa.feature.rms(
    y=y, frame_length=512, hop_length=160
)[0]

rms_times = librosa.frames_to_time(
    np.arange(len(rms)), sr=sr, hop_length=160
)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

times = np.arange(len(y)) / sr
axes[0].plot(times, y, linewidth=0.5)
axes[0].set_title("Waveform and speech segments")
axes[0].set_ylabel("Amplitude")

for start, end in energy_intervals:
    axes[0].axvspan(start/sr, end/sr, alpha=0.18)

for seg in vad_intervals:
    axes[0].axvline(seg["start"]/sr, linestyle="--")
    axes[0].axvline(seg["end"]/sr, linestyle="--")

axes[1].plot(rms_times, rms, linewidth=1)
axes[1].set_title("Short-time RMS energy")
axes[1].set_ylabel("RMS")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_xlim(0, duration)

plt.tight_layout()
plt.savefig(OUTPUT, dpi=160)
plt.show()

print("=== SEGMENTATION DETAILS ===")
print("Audio ID:", TARGET_ID)
print("Duration:", round(duration, 3))

print("\nLibrosa speech intervals:")
for start, end in energy_intervals:
    print(round(start/sr, 3), "-", round(end/sr, 3))

print("\nSilero speech intervals:")
for seg in vad_intervals:
    print(
        round(seg["start"]/sr, 3),
        "-",
        round(seg["end"]/sr, 3)
    )

print("\nFigure saved:", OUTPUT)


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"

OUT = PROJECT / "features/acoustic/hybrid_pause_diagnostics_000700008.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

TARGET_ID = "000700008"
pf = pq.ParquetFile(TRAIN)

audio_bytes = None

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        if Path(audio["path"]).stem == TARGET_ID:
            audio_bytes = audio["bytes"]
            break

    if audio_bytes is not None:
        break

assert audio_bytes is not None

y, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32")

if "model" not in globals() or "get_speech_timestamps" not in globals():
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]

vad = get_speech_timestamps(
    torch.from_numpy(y),
    model,
    sampling_rate=sr,
    threshold=0.5,
    min_speech_duration_ms=100,
    min_silence_duration_ms=200,
    speech_pad_ms=0
)

assert len(vad) > 0

padding = int(0.2 * sr)

start = max(0, vad[0]["start"] - padding)
end = min(len(y), vad[-1]["end"] + padding)

segment = y[start:end]

intervals = librosa.effects.split(
    segment,
    top_db=30,
    frame_length=512,
    hop_length=160
)

# Convert local intervals to original timeline
intervals_global = [
    (int(a + start), int(b + start))
    for a, b in intervals
]

records = []

for idx, (a, b) in enumerate(intervals_global):
    length_sec = (b - a) / sr

    records.append({
        "segment_id": idx + 1,
        "start_sec": round(a / sr, 3),
        "end_sec": round(b / sr, 3),
        "duration_sec": round(length_sec, 3),
        "shorter_than_50ms": length_sec < 0.05
    })

df = pd.DataFrame(records)
df.to_csv(OUT, index=False)

# Inspect gaps only between substantial active segments
substantial = [
    (a, b) for a, b in intervals_global
    if (b - a) / sr >= 0.05
]

gaps = [
    (substantial[i + 1][0] - substantial[i][1]) / sr
    for i in range(len(substantial) - 1)
]

print("=== HYBRID SEGMENTATION ===")
print("Silero interval:", round(vad[0]["start"]/sr, 3),
      "-", round(vad[-1]["end"]/sr, 3))
print("Padded interval:", round(start/sr, 3),
      "-", round(end/sr, 3))
print("\nEnergy segments:")
print(df.to_string(index=False))

print("\n=== INTERNAL GAPS ===")
print("All gaps:", [round(g, 3) for g in gaps])
print("Pauses >= 100ms:", sum(g >= 0.1 for g in gaps))
print("Pauses >= 200ms:", sum(g >= 0.2 for g in gaps))

print("\nSaved:", OUT)


In [ ]:

import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
OUT = PROJECT / "features/acoustic/hybrid_validation_5.csv"

OUT.parent.mkdir(parents=True, exist_ok=True)

pilot = pd.read_csv(PILOT, dtype={"audio_id": "string"})
target_ids = set(pilot["audio_id"].head(5))

if "model" not in globals() or "get_speech_timestamps" not in globals():
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]

pf = pq.ParquetFile(TRAIN)
records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        audio_id = Path(audio["path"]).stem
        if audio_id not in target_ids:
            continue

        found.add(audio_id)
        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        duration = len(y) / sr

        vad = get_speech_timestamps(
            torch.from_numpy(y),
            model,
            sampling_rate=sr,
            threshold=0.5,
            min_speech_duration_ms=100,
            min_silence_duration_ms=200,
            speech_pad_ms=0
        )

        if not vad:
            records.append({
                "audio_id": audio_id,
                "status": "no_speech_detected"
            })
            continue

        pad = int(0.2 * sr)
        start = max(0, vad[0]["start"] - pad)
        end = min(len(y), vad[-1]["end"] + pad)

        intervals = librosa.effects.split(
            y[start:end],
            top_db=30,
            frame_length=512,
            hop_length=160
        )

        intervals = [
            (a + start, b + start)
            for a, b in intervals
            if (b - a) / sr >= 0.05
        ]

        speech_sec = sum(
            (b - a) / sr for a, b in intervals
        )

        gaps = [
            (intervals[i + 1][0] - intervals[i][1]) / sr
            for i in range(len(intervals) - 1)
        ]

        pauses_100 = [g for g in gaps if g >= 0.1]
        pauses_200 = [g for g in gaps if g >= 0.2]

        records.append({
            "audio_id": audio_id,
            "status": "ok",
            "audio_duration_sec": round(duration, 3),
            "hybrid_speech_sec": round(speech_sec, 3),
            "speech_segments": len(intervals),
            "pause_count_100ms": len(pauses_100),
            "pause_count_200ms": len(pauses_200),
            "pause_duration_200ms": round(sum(pauses_200), 3),
            "leading_silence_sec": round(
                intervals[0][0] / sr, 3
            ) if intervals else np.nan,
            "trailing_silence_sec": round(
                (len(y) - intervals[-1][1]) / sr, 3
            ) if intervals else np.nan
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(records)
result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)

result = result.sort_values("pilot_index")
result.to_csv(OUT, index=False)

print("=== HYBRID VALIDATION ===")
print(result.to_string(index=False))

print("\n=== SUMMARY ===")
print("Processed:", len(result))
print("Successful:", (result["status"] == "ok").sum())
print("Failed:", (result["status"] != "ok").sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\nSaved:", OUT)


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

selected = pilot.head(5)
target_ids = set(selected["audio_id"])

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(group_id, columns=["audio"])

    for item in table.column("audio").to_pylist():
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == target_ids

print("=== AUDIO LISTENING AUDIT ===")

for _, row in selected.iterrows():
    audio_id = row["audio_id"]
    y, sr = audio_data[audio_id]

    print(
        f"\nAudio: {audio_id} | "
        f"Duration: {len(y)/sr:.2f}s"
    )
    print("Expected text:", row["text"])

    display(Audio(y, rate=sr, normalize=False))

print("\nAll 5 audio players displayed.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import json
import zipfile
import hashlib

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
META = PROJECT / "data/metadata"
REPORTS = PROJECT / "reports"
CHECK = PROJECT / "checkpoints"

for folder in [META, REPORTS, CHECK]:
    folder.mkdir(parents=True, exist_ok=True)

observations = [
    {
        "audio_id": "000700008",
        "audible_internal_pause_200ms": "not_observed",
        "leading_noise": True,
        "trailing_noise": True,
        "notes": "Short initial noise; no noticeable internal stop; short trailing silence and noise"
    },
    {
        "audio_id": "000700010",
        "audible_internal_pause_200ms": "not_observed",
        "leading_noise": None,
        "trailing_noise": None,
        "notes": "No noticeable internal stop; short trailing silence"
    },
    {
        "audio_id": "000700012",
        "audible_internal_pause_200ms": "not_assessed",
        "leading_noise": None,
        "trailing_noise": True,
        "notes": "Breathing-like noise after speech"
    },
    {
        "audio_id": "000700015",
        "audible_internal_pause_200ms": "not_assessed",
        "leading_noise": True,
        "trailing_noise": None,
        "notes": "Mouth-opening-like noise before speech"
    },
    {
        "audio_id": "000700019",
        "audible_internal_pause_200ms": "not_assessed",
        "leading_noise": None,
        "trailing_noise": None,
        "notes": "No unusual issue noticed"
    }
]

df = pd.DataFrame(observations)

AUDIT = META / "audio_listening_audit_5.csv"
df.to_csv(AUDIT, index=False, encoding="utf-8")

# Preserve the evidence and its limitations
report = """# STEP 2 - Preliminary Listening Audit

Dataset: Speechocean762
Audios checked: 5
Auditor: Researcher (informal listening)

## Findings
- No noticeable internal stops reported for 000700008 and 000700010.
- Initial/trailing non-speech noises were reported.
- Remaining recordings received general qualitative observations.
- Hybrid method detected zero internal pauses >= 200 ms in all five.

## Important limitations
- This is not expert-validated pause ground truth.
- The observer did not timestamp pauses manually.
- 'Not assessed' must not be interpreted as 'no pause'.
- Current parameters have not been validated across speakers.
- Do not claim 100% segmentation accuracy.

## Next task
Prepare an expanded, speaker-diverse quality audit before
finalizing the 400-audio acoustic extraction configuration.
"""

(REPORTS / "step02_listening_audit.md").write_text(
    report, encoding="utf-8"
)

progress_file = CHECK / "research_progress.json"
progress = json.loads(
    progress_file.read_text(encoding="utf-8")
)

progress["current_step"] = 2
progress["step2_listening_audit_count"] = 5
progress["step2_listening_audit_file"] = str(AUDIT)
progress["step2_segmentation_status"] = "preliminary_validation"
progress["next_task"] = "Expanded speaker-diverse segmentation QA"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

progress_file.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

ZIP = Path("/kaggle/working/asr_fluency_step02_checkpoint.zip")

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [
        PROJECT / "data/metadata",
        PROJECT / "features/acoustic",
        REPORTS,
        CHECK
    ]:
        if folder.exists():
            for file in folder.rglob("*"):
                if file.is_file():
                    z.write(file, file.relative_to(PROJECT))

with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None

sha = hashlib.sha256(ZIP.read_bytes()).hexdigest()

print("=== STEP 2 CHECKPOINT ===")
print("Listening audit rows:", len(df))
print("Audit saved:", AUDIT)
print("ZIP:", ZIP)
print("ZIP size (MB):", round(ZIP.stat().st_size / 1048576, 3))
print("ZIP SHA256:", sha)
print("ZIP integrity: PASS")
print("Next task:", progress["next_task"])


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
OUT = PROJECT / "data/metadata/expanded_qa_selection_15.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

SEED = 42

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(pilot) == 400
assert pilot["audio_id"].is_unique

# Exclude the five recordings already reviewed
previous_ids = set(pilot.head(5)["audio_id"])
available = pilot[
    ~pilot["audio_id"].isin(previous_ids)
].copy()

if OUT.exists():
    selected = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )
    print("Existing QA selection restored.")
else:
    rng = np.random.default_rng(SEED)

    # Select from three fluency bands
    available["fluency_band"] = pd.cut(
        available["fluency"],
        bins=[0, 6, 8, 10],
        labels=["low", "medium", "high"]
    )

    selections = []

    for band in ["low", "medium", "high"]:
        group = available[
            available["fluency_band"] == band
        ].copy()

        # Randomize speaker order, then choose one
        # recording per speaker where possible
        speakers = rng.permutation(
            sorted(group["speaker"].unique())
        )

        chosen = []

        for speaker in speakers:
            candidates = group[
                group["speaker"] == speaker
            ]

            idx = rng.choice(candidates.index.to_numpy())
            chosen.append(group.loc[idx])

            if len(chosen) == 5:
                break

        if len(chosen) != 5:
            raise RuntimeError(
                f"Insufficient speaker diversity in {band}"
            )

        selections.extend(chosen)

    selected = pd.DataFrame(selections)

    selected = selected.sort_values(
        ["fluency_band", "speaker", "audio_id"]
    ).reset_index(drop=True)

    selected.insert(
        0, "qa_index",
        range(1, len(selected) + 1)
    )

    assert len(selected) == 15
    assert selected["audio_id"].is_unique
    assert not set(selected["audio_id"]) & previous_ids

    OUT.parent.mkdir(parents=True, exist_ok=True)
    selected.to_csv(OUT, index=False)

print("=== EXPANDED QA SELECTION ===")
print("Selected audio:", len(selected))
print("Unique speakers:", selected["speaker"].nunique())

print("\nFluency bands:")
print(selected["fluency_band"].value_counts())

print("\nDuration statistics:")
print(selected["duration_sec"].describe())

print("\n=== SELECTED RECORDINGS ===")
print(
    selected[
        [
            "qa_index", "audio_id", "speaker",
            "fluency", "duration_sec", "text"
        ]
    ].to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_expanded_qa_count"] = len(selected)
progress["step2_expanded_qa_file"] = str(OUT)
progress["next_task"] = "Inspect expanded QA audio and segmentation"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nSaved:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
QA = PROJECT / "data/metadata/expanded_qa_selection_15.csv"
OUT = PROJECT / "features/acoustic/expanded_hybrid_diagnostics_15.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT.parent.mkdir(parents=True, exist_ok=True)

qa = pd.read_csv(
    QA,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(qa) == 15
assert qa["audio_id"].is_unique

target_ids = set(qa["audio_id"])

# Load Silero VAD if not already available
if "model" not in globals() or "get_speech_timestamps" not in globals():
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]

pf = pq.ParquetFile(TRAIN)
records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        duration = len(y) / sr

        vad = get_speech_timestamps(
            torch.from_numpy(y),
            model,
            sampling_rate=sr,
            threshold=0.5,
            min_speech_duration_ms=100,
            min_silence_duration_ms=200,
            speech_pad_ms=0
        )

        if not vad:
            records.append({
                "audio_id": audio_id,
                "status": "no_speech_detected"
            })
            continue

        pad = int(0.2 * sr)
        start = max(0, vad[0]["start"] - pad)
        end = min(len(y), vad[-1]["end"] + pad)

        intervals = librosa.effects.split(
            y[start:end],
            top_db=30,
            frame_length=512,
            hop_length=160
        )

        intervals = [
            (int(a + start), int(b + start))
            for a, b in intervals
            if (b - a) / sr >= 0.05
        ]

        if not intervals:
            records.append({
                "audio_id": audio_id,
                "status": "no_energy_segments"
            })
            continue

        speech_sec = sum(
            (b - a) / sr for a, b in intervals
        )

        gaps = [
            (intervals[i + 1][0] - intervals[i][1]) / sr
            for i in range(len(intervals) - 1)
        ]

        pauses_100 = [g for g in gaps if g >= 0.1]
        pauses_200 = [g for g in gaps if g >= 0.2]

        records.append({
            "audio_id": audio_id,
            "status": "ok",
            "duration_sec": round(duration, 3),
            "hybrid_speech_sec": round(speech_sec, 3),
            "speech_segments": len(intervals),
            "pause_count_100ms": len(pauses_100),
            "pause_count_200ms": len(pauses_200),
            "pause_duration_200ms": round(
                sum(pauses_200), 3
            ),
            "leading_silence_sec": round(
                intervals[0][0] / sr, 3
            ),
            "trailing_silence_sec": round(
                (len(y) - intervals[-1][1]) / sr, 3
            )
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(records)

result = qa[
    ["qa_index", "audio_id", "speaker", "fluency"]
].merge(
    result,
    on="audio_id",
    how="left",
    validate="one_to_one"
)

result = result.sort_values("qa_index")
result.to_csv(OUT, index=False)

print("=== EXPANDED HYBRID QA ===")
print(result.to_string(index=False))

print("\n=== SUMMARY ===")
print("Processed:", len(result))
print("Successful:", (result["status"] == "ok").sum())
print("Failed:", (result["status"] != "ok").sum())
print("Audios with 100ms pauses:",
      (result["pause_count_100ms"] > 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\nSaved:", OUT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step2_expanded_hybrid_processed"] = len(result)
progress["step2_expanded_hybrid_successful"] = int(
    (result["status"] == "ok").sum()
)
progress["next_task"] = "Expanded listening audit"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
QA = PROJECT / "data/metadata/expanded_qa_selection_15.csv"
DIAG = PROJECT / "features/acoustic/expanded_hybrid_diagnostics_15.csv"
OUT = PROJECT / "data/metadata/targeted_listening_audit_5.csv"

target_ids = [
    "000700065",
    "096140018",
    "096190016",
    "000700109",
    "001450142"
]

qa = pd.read_csv(QA, dtype={"audio_id": "string"})
diag = pd.read_csv(DIAG, dtype={"audio_id": "string"})

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert len(audio_data) == len(target_ids)

records = []

print("=== TARGETED LISTENING AUDIT ===")

for audio_id in target_ids:
    row = qa.loc[qa["audio_id"] == audio_id].iloc[0]
    d = diag.loc[diag["audio_id"] == audio_id].iloc[0]
    y, sr = audio_data[audio_id]

    print("\n" + "=" * 55)
    print("Audio ID:", audio_id)
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(len(y) / sr, 3))
    print("Hybrid 200ms pauses:", d["pause_count_200ms"])
    print("Hybrid pause duration:", d["pause_duration_200ms"])

    display(Audio(y, rate=sr, normalize=False))

    records.append({
        "audio_id": audio_id,
        "fluency": row["fluency"],
        "hybrid_pause_count_200ms": d["pause_count_200ms"],
        "hybrid_pause_duration_200ms": d["pause_duration_200ms"],
        "manual_pause_observation": "pending",
        "noise_observation": "pending",
        "notes": ""
    })

audit = pd.DataFrame(records)
OUT.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUT, index=False)

print("\nAudit template saved:", OUT)


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
AUDIT = PROJECT / "data/metadata/targeted_listening_audit_5.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Audit template not found"

audit = pd.read_csv(AUDIT, dtype={"audio_id": "string"})

annotations = {
    "000700065": {
        "manual_pause_observation": "internal_pauses_audible",
        "noise_observation": "not_assessed",
        "notes": (
            "Internal pauses audible, especially before FANS. "
            "FANS pronounced approximately fffaans."
        )
    },
    "096140018": {
        "manual_pause_observation": "internal_pauses_audible",
        "noise_observation": "not_assessed",
        "notes": (
            "Internal pauses audible. "
            "DEAD sounded approximately like daddi."
        )
    },
    "096190016": {
        "manual_pause_observation": "frequent_internal_pauses",
        "noise_observation": "not_assessed",
        "notes": (
            "Pauses after many words; several pronunciation "
            "deviations reported: ASKED, AS, PUT, HER."
        )
    },
    "000700109": {
        "manual_pause_observation": "internal_pause_audible",
        "noise_observation": "not_assessed",
        "notes": "Audible pause between THE and YET."
    },
    "001450142": {
        "manual_pause_observation": "not_assessed",
        "noise_observation": "leading_breath_like_noise",
        "notes": "Breathing-like noise at beginning."
    }
}

assert set(audit["audio_id"]) == set(annotations)

for audio_id, values in annotations.items():
    mask = audit["audio_id"] == audio_id

    for column, value in values.items():
        audit.loc[mask, column] = value

audit["audit_type"] = "informal_researcher_listening"
audit["manual_pause_timestamps_available"] = False

audit.to_csv(AUDIT, index=False, encoding="utf-8")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_targeted_audit_count"] = len(audit)
progress["step2_targeted_audit_status"] = "completed"
progress["next_task"] = "Listen to remaining 10 expanded QA recordings"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== TARGETED AUDIT SAVED ===")
print(audit[
    [
        "audio_id",
        "manual_pause_observation",
        "noise_observation",
        "notes"
    ]
].to_string(index=False))

print("\nAudit rows:", len(audit))
print("Pending observations:",
      (audit["manual_pause_observation"] == "pending").sum())
print("Saved:", AUDIT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
QA = PROJECT / "data/metadata/expanded_qa_selection_15.csv"
DIAG = PROJECT / "features/acoustic/expanded_hybrid_diagnostics_15.csv"
PREVIOUS = PROJECT / "data/metadata/targeted_listening_audit_5.csv"
OUT = PROJECT / "data/metadata/remaining_listening_audit_10.csv"

qa = pd.read_csv(QA, dtype={"audio_id": "string"})
diag = pd.read_csv(DIAG, dtype={"audio_id": "string"})
previous = pd.read_csv(
    PREVIOUS,
    dtype={"audio_id": "string"}
)

completed_ids = set(previous["audio_id"])

remaining = qa[
    ~qa["audio_id"].isin(completed_ids)
].copy().sort_values("qa_index")

assert len(remaining) == 10
assert remaining["audio_id"].is_unique

target_ids = set(remaining["audio_id"])

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id,
        columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == target_ids

records = []

print("=== REMAINING LISTENING AUDIT ===")

for _, row in remaining.iterrows():
    audio_id = row["audio_id"]
    d = diag.loc[
        diag["audio_id"] == audio_id
    ].iloc[0]

    y, sr = audio_data[audio_id]

    print("\n" + "=" * 55)
    print("QA index:", row["qa_index"])
    print("Audio ID:", audio_id)
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(len(y) / sr, 3))
    print("Hybrid 200ms pauses:", d["pause_count_200ms"])
    print("Hybrid pause duration:",
          d["pause_duration_200ms"])

    display(Audio(y, rate=sr, normalize=False))

    records.append({
        "qa_index": row["qa_index"],
        "audio_id": audio_id,
        "fluency": row["fluency"],
        "hybrid_pause_count_200ms": d["pause_count_200ms"],
        "hybrid_pause_duration_200ms": d["pause_duration_200ms"],
        "manual_pause_observation": "pending",
        "noise_observation": "pending",
        "pronunciation_observation": "pending",
        "notes": ""
    })

audit = pd.DataFrame(records)

# Explicit string dtypes avoid future pandas warnings
for col in [
    "manual_pause_observation",
    "noise_observation",
    "pronunciation_observation",
    "notes"
]:
    audit[col] = audit[col].astype("string")

OUT.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUT, index=False)

print("\n=== AUDIT SUMMARY ===")
print("Audio players displayed:", len(audio_data))
print("Remaining audit rows:", len(audit))
print("Pending observations:", len(audit))
print("Saved:", OUT)


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
AUDIT = PROJECT / "data/metadata/remaining_listening_audit_10.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Audit file not found"

audit = pd.read_csv(AUDIT, dtype={"audio_id": "string"})

annotations = {
    "009600270": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    ),
    "024020175": (
        "no_noticeable_pause", "not_assessed",
        "natural_pronunciation", "Native-like pronunciation"
    ),
    "030210137": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    ),
    "069120123": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    ),
    "060100128": (
        "one_or_two_pauses", "not_assessed",
        "not_assessed",
        "Pauses around TALKING-AWAY and AWAY-THE TABLES"
    ),
    "096110018": (
        "no_noticeable_pause", "not_assessed",
        "pronunciation_deviation",
        "Pronunciation errors audible; no pauses noticed"
    ),
    "020590276": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    ),
    "024020051": (
        "one_natural_pause", "not_assessed",
        "natural_pronunciation",
        "Native-like pronunciation; one natural pause"
    ),
    "034230115": (
        "one_natural_pause", "leading_breath_like_noise",
        "no_issue_noticed",
        "Natural pause; breathing-like noise at beginning"
    ),
    "054140039": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    )
}

assert set(audit["audio_id"]) == set(annotations)

columns = [
    "manual_pause_observation",
    "noise_observation",
    "pronunciation_observation",
    "notes"
]

# Explicit string dtype prevents the previous FutureWarning
for col in columns:
    audit[col] = audit[col].astype("string")

for audio_id, values in annotations.items():
    mask = audit["audio_id"] == audio_id
    for col, value in zip(columns, values):
        audit.loc[mask, col] = value

audit["audit_type"] = "informal_researcher_listening"
audit["manual_pause_timestamps_available"] = False

audit.to_csv(AUDIT, index=False, encoding="utf-8")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_remaining_audit_count"] = len(audit)
progress["step2_remaining_audit_status"] = "completed"
progress["step2_total_listening_audit_count"] = 20
progress["next_task"] = (
    "Review segmentation QA and finalize pilot extraction configuration"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== REMAINING AUDIT SAVED ===")
print(
    audit[
        ["audio_id", "manual_pause_observation",
         "noise_observation", "notes"]
    ].to_string(index=False)
)

print("\nAudit rows:", len(audit))
print("Pending observations:",
      (audit["manual_pause_observation"] == "pending").sum())
print("Total listening-audit recordings:",
      progress["step2_total_listening_audit_count"])
print("Saved:", AUDIT)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
CONFIG_DIR = PROJECT / "configs"
REPORTS = PROJECT / "reports"
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

config = {
    "configuration_name": "pilot_acoustic_v1",
    "status": "frozen_for_pilot_not_final_validation",
    "sample_rate_hz": 16000,
    "silero_vad": {
        "threshold": 0.5,
        "min_speech_duration_ms": 100,
        "min_silence_duration_ms": 200,
        "speech_pad_ms": 0,
        "hybrid_padding_ms": 200
    },
    "energy_segmentation": {
        "method": "librosa.effects.split",
        "top_db": 30,
        "frame_length": 512,
        "hop_length": 160,
        "min_active_segment_ms": 50
    },
    "pitch": {
        "method": "librosa.pyin",
        "fmin_hz": 75,
        "fmax_hz": 500,
        "frame_length": 1024,
        "hop_length": 256,
        "fill_na": "nan",
        "statistics": ["mean", "sd", "voiced_ratio"]
    },
    "pause_thresholds_ms": [100, 200],
    "main_pause_threshold_ms": 200,
    "features": [
        "duration_sec",
        "speech_duration_sec",
        "speech_ratio",
        "pause_count_100ms",
        "pause_count_200ms",
        "pause_duration_200ms",
        "pause_ratio_200ms",
        "leading_silence_sec",
        "trailing_silence_sec",
        "pitch_mean_hz",
        "pitch_sd_hz",
        "voiced_ratio",
        "rms_mean",
        "rms_sd"
    ],
    "limitations": [
        "No expert timestamped pause reference",
        "Prompt text is not a verbatim transcript",
        "Detected pauses do not necessarily indicate hesitation",
        "Silero boundaries may omit low-energy speech",
        "Pitch estimates may contain octave errors",
        "Configuration is frozen for pilot consistency only"
    ]
}

CONFIG = CONFIG_DIR / "pilot_acoustic_v1.json"
CONFIG.write_text(
    json.dumps(config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

report = """# STEP 2 - Pilot Acoustic Configuration

## Dataset
Speechocean762, official train split only.

## Pilot
400 recordings from 20 speakers.

## Segmentation
Silero VAD followed by energy-based segmentation
within the padded outer speech boundaries.

## Quality assurance
20 recordings informally reviewed by researcher.
No expert timestamped annotation was conducted.

## Pitch
librosa.pyin voiced-frame statistics.

## Feature selection
14 acoustic features are defined in pilot_acoustic_v1.json.

## Limitations
The QA sample is not sufficient to establish overall
segmentation accuracy. Pause occurrence and disfluency
are not equivalent.

## Next
Extract acoustic features for the 400-audio pilot,
saving batches of 100 with stable identifiers.
"""

(REPORTS / "step02_configuration_report.md").write_text(
    report, encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_config_frozen"] = True
progress["step2_config_name"] = "pilot_acoustic_v1"
progress["step2_pilot_target_audio"] = 400
progress["step2_pilot_processed_audio"] = 0
progress["next_task"] = "Extract acoustic features for pilot rows 0001-0100"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== PILOT CONFIGURATION FROZEN ===")
print("Config:", CONFIG)
print("Features:", len(config["features"]))
print("QA recordings:", 20)
print("Pilot target:", 400)
print("Configuration status:", config["status"])
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "features/acoustic/pilot_acoustic_features_0001-0100.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["configuration_name"] == "pilot_acoustic_v1"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.sort_values("pilot_index").head(100).copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(1, 101))
assert batch["audio_id"].is_unique

feature_columns = config["features"]
expected_ids = set(batch["audio_id"])

if OUT.exists():
    existing = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )

    valid = (
        len(existing) == 100
        and existing["audio_id"].is_unique
        and set(existing["audio_id"]) == expected_ids
        and all(c in existing.columns for c in feature_columns)
    )

    assert valid, "Existing CSV failed validation; do not overwrite"

    result = existing
    print("Existing validated batch reused.")

else:
    if "model" not in globals() or "get_speech_timestamps" not in globals():
        model, utils = torch.hub.load(
            "snakers4/silero-vad",
            "silero_vad",
            trust_repo=True
        )
        get_speech_timestamps = utils[0]

    model.eval()

    pf = pq.ParquetFile(TRAIN)
    audio_data = {}

    for group_id in range(pf.metadata.num_row_groups):
        audios = pf.read_row_group(
            group_id, columns=["audio"]
        ).column("audio").to_pylist()

        for item in audios:
            audio_id = Path(item["path"]).stem

            if audio_id in expected_ids:
                audio_data[audio_id] = item["bytes"]

        if len(audio_data) == 100:
            break

    assert set(audio_data) == expected_ids

    records = []

    with torch.no_grad():

        for n, (_, row) in enumerate(batch.iterrows(), start=1):
            audio_id = row["audio_id"]

            y, sr = sf.read(
                io.BytesIO(audio_data[audio_id]),
                dtype="float32"
            )

            assert sr == 16000
            assert y.ndim == 1

            duration = len(y) / sr

            sc = config["silero_vad"]
            ec = config["energy_segmentation"]
            pc = config["pitch"]

            vad = get_speech_timestamps(
                torch.from_numpy(y),
                model,
                sampling_rate=sr,
                threshold=sc["threshold"],
                min_speech_duration_ms=sc["min_speech_duration_ms"],
                min_silence_duration_ms=sc["min_silence_duration_ms"],
                speech_pad_ms=sc["speech_pad_ms"]
            )

            if vad:
                pad = int(sc["hybrid_padding_ms"] * sr / 1000)

                start = max(0, vad[0]["start"] - pad)
                end = min(len(y), vad[-1]["end"] + pad)

                intervals = librosa.effects.split(
                    y[start:end],
                    top_db=ec["top_db"],
                    frame_length=ec["frame_length"],
                    hop_length=ec["hop_length"]
                )

                intervals = [
                    (int(a + start), int(b + start))
                    for a, b in intervals
                    if (b-a)/sr >= ec["min_active_segment_ms"]/1000
                ]
            else:
                intervals = []

            speech_sec = sum(
                (b-a)/sr for a,b in intervals
            )

            gaps = [
                (intervals[i+1][0]-intervals[i][1])/sr
                for i in range(len(intervals)-1)
            ]

            pauses100 = [g for g in gaps if g >= 0.1]
            pauses200 = [g for g in gaps if g >= 0.2]

            f0, voiced_flag, _ = librosa.pyin(
                y,
                sr=sr,
                fmin=pc["fmin_hz"],
                fmax=pc["fmax_hz"],
                frame_length=pc["frame_length"],
                hop_length=pc["hop_length"],
                fill_na=np.nan
            )

            voiced_f0 = f0[
                voiced_flag & np.isfinite(f0)
            ]

            rms = librosa.feature.rms(
                y=y,
                frame_length=1024,
                hop_length=256
            )[0]

            records.append({
                "pilot_index": int(row["pilot_index"]),
                "audio_id": audio_id,
                "speaker": row["speaker"],
                "fluency": row["fluency"],

                "duration_sec": duration,
                "speech_duration_sec": speech_sec,
                "speech_ratio": speech_sec / duration,

                "pause_count_100ms": len(pauses100),
                "pause_count_200ms": len(pauses200),
                "pause_duration_200ms": sum(pauses200),
                "pause_ratio_200ms": sum(pauses200) / duration,

                "leading_silence_sec": (
                    intervals[0][0] / sr
                    if intervals else np.nan
                ),
                "trailing_silence_sec": (
                    (len(y)-intervals[-1][1])/sr
                    if intervals else np.nan
                ),

                "pitch_mean_hz": (
                    float(np.mean(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "pitch_sd_hz": (
                    float(np.std(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "voiced_ratio": len(voiced_f0) / len(f0),

                "rms_mean": float(np.mean(rms)),
                "rms_sd": float(np.std(rms))
            })

            if n % 10 == 0:
                print(f"Processed {n}/100")

    result = pd.DataFrame(records)
    result.to_csv(OUT, index=False)

# Validate the complete batch
assert len(result) == 100
assert result["pilot_index"].tolist() == list(range(1, 101))
assert result["audio_id"].is_unique
assert set(result["audio_id"]) == expected_ids

assert result["duration_sec"].gt(0).all()
assert result["speech_duration_sec"].ge(0).all()
assert (
    result["speech_duration_sec"]
    <= result["duration_sec"] + 1e-5
).all()

assert result["speech_ratio"].between(0, 1.00001).all()
assert result["voiced_ratio"].between(0, 1).all()

sha256 = hashlib.sha256(OUT.read_bytes()).hexdigest()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_pilot_processed_audio"] = 100
progress["step2_last_batch"] = "0001-0100"
progress["step2_last_batch_sha256"] = sha256
progress["next_task"] = "Validate first 100 acoustic features"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ACOUSTIC BATCH 0001-0100 ===")
print("Rows:", len(result))
print("Speakers:", result["speaker"].nunique())
print("Features:", len(feature_columns))
print("Missing pitch values:", result["pitch_mean_hz"].isna().sum())
print("Zero speech duration:",
      (result["speech_duration_sec"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\n=== FEATURE SUMMARY ===")
print(result[feature_columns].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", sha256)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = PROJECT / "features/acoustic/pilot_acoustic_features_0001-0100.csv"
OUT = PROJECT / "reports/acoustic_qc_batch_0001-0100.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 100
assert df["audio_id"].is_unique

# Screening thresholds, not validated error criteria
df["flag_long_pause"] = df["pause_duration_200ms"] >= 1.5
df["flag_low_speech_ratio"] = df["speech_ratio"] < 0.4
df["flag_low_voiced_ratio"] = df["voiced_ratio"] < 0.2
df["flag_high_pitch_sd"] = df["pitch_sd_hz"] > 70

# Basic numerical consistency checks
df["flag_invalid_duration"] = (
    (df["duration_sec"] <= 0)
    | (df["speech_duration_sec"] < 0)
    | (df["speech_duration_sec"] > df["duration_sec"] + 1e-5)
)

df["flag_invalid_pause"] = (
    (df["pause_duration_200ms"] < 0)
    | (df["pause_duration_200ms"] > df["duration_sec"] + 1e-5)
    | (df["pause_count_200ms"] < 0)
    | (df["pause_count_200ms"] > df["pause_count_100ms"])
)

flag_cols = [c for c in df.columns if c.startswith("flag_")]

df["qc_flag_count"] = df[flag_cols].sum(axis=1)

flagged = df[df["qc_flag_count"] > 0].copy()
flagged = flagged.sort_values(
    ["qc_flag_count", "pause_duration_200ms"],
    ascending=[False, False]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
flagged.to_csv(OUT, index=False)

print("=== ACOUSTIC QC BATCH 0001-0100 ===")
print("Total recordings:", len(df))
print("Flagged recordings:", len(flagged))

print("\n=== FLAG COUNTS ===")
for col in flag_cols:
    print(col, ":", int(df[col].sum()))

print("\n=== TOP FLAGGED RECORDINGS ===")
display_cols = [
    "pilot_index", "audio_id", "speaker", "fluency",
    "duration_sec", "speech_ratio",
    "pause_count_200ms", "pause_duration_200ms",
    "pitch_mean_hz", "pitch_sd_hz", "voiced_ratio",
    "qc_flag_count"
]

print(flagged[display_cols].head(20).round(3).to_string(index=False))

print("\n=== TOP 5 LONGEST PAUSE DURATIONS ===")
print(
    df.nlargest(5, "pause_duration_200ms")[
        ["pilot_index", "audio_id", "speaker", "fluency",
         "duration_sec", "pause_duration_200ms",
         "pause_count_200ms"]
    ].round(3).to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_batch1_qc_completed"] = True
progress["step2_batch1_qc_flagged"] = len(flagged)
progress["next_task"] = "Review QC flags before batch 0101-0200"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nQC CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
FEATURES = PROJECT / "features/acoustic/pilot_acoustic_features_0001-0100.csv"
OUT = PROJECT / "data/metadata/long_pause_audit_4.csv"

target_ids = [
    "001450116",
    "001450051",
    "001450109",
    "001450094"
]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"}
)

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id,
        columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == set(target_ids)

records = []

print("=== LONG PAUSE LISTENING AUDIT ===")

for audio_id in target_ids:
    row = pilot.loc[
        pilot["audio_id"] == audio_id
    ].iloc[0]

    feat = features.loc[
        features["audio_id"] == audio_id
    ].iloc[0]

    y, sr = audio_data[audio_id]

    print("\n" + "=" * 55)
    print("Audio ID:", audio_id)
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(len(y) / sr, 3))
    print("Hybrid pause count:",
          int(feat["pause_count_200ms"]))
    print("Hybrid pause duration:",
          round(feat["pause_duration_200ms"], 3))

    display(Audio(y, rate=sr, normalize=False))

    records.append({
        "audio_id": audio_id,
        "fluency": row["fluency"],
        "hybrid_pause_count_200ms": int(
            feat["pause_count_200ms"]
        ),
        "hybrid_pause_duration_200ms": float(
            feat["pause_duration_200ms"]
        ),
        "manual_pause_observation": "pending",
        "notes": ""
    })

audit = pd.DataFrame(records)

for col in ["manual_pause_observation", "notes"]:
    audit[col] = audit[col].astype("string")

OUT.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUT, index=False)

print("\n=== AUDIT SUMMARY ===")
print("Audio players:", len(audio_data))
print("Audit rows:", len(audit))
print("Saved:", OUT)


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

AUDIT = PROJECT / "data/metadata/long_pause_audit_4.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Long-pause audit CSV not found"

audit = pd.read_csv(
    AUDIT,
    dtype={"audio_id": "string"}
)

annotations = {
    "001450116": {
        "manual_pause_observation": "four_pauses_audible",
        "notes": (
            "Four pauses reported after BUTTER, WANTS, "
            "TO, and BE A GOOD. No other unusual issue noticed."
        )
    },
    "001450051": {
        "manual_pause_observation": "two_pauses_audible",
        "notes": (
            "Slow pronunciation. Long pause after LIKES "
            "and another pause after RETURNS. "
            "RETURNS sounded approximately like RENTURS."
        )
    },
    "001450109": {
        "manual_pause_observation": "two_pauses_audible",
        "notes": (
            "Pause after WASHED, followed by an extra "
            "a-like utterance, then another pause."
        )
    },
    "001450094": {
        "manual_pause_observation": "two_pauses_audible",
        "notes": (
            "Long pause after THE and another after ME. "
            "DAVID sounded approximately like DAIVI."
        )
    }
}

assert len(audit) == 4
assert audit["audio_id"].is_unique
assert set(audit["audio_id"]) == set(annotations)

for column in ["manual_pause_observation", "notes"]:
    audit[column] = audit[column].astype("string")

for audio_id, values in annotations.items():
    mask = audit["audio_id"] == audio_id

    for column, value in values.items():
        audit.loc[mask, column] = value

audit["audit_type"] = "informal_researcher_listening"
audit["manual_pause_timestamps_available"] = False
audit["pause_duration_manually_verified"] = False

audit.to_csv(
    AUDIT,
    index=False,
    encoding="utf-8"
)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step2_long_pause_audit_count"] = 4
progress["step2_long_pause_audit_status"] = "completed"
progress["step2_batch1_qc_review_status"] = "preliminary_pass"
progress["next_task"] = (
    "Extract acoustic features for pilot rows 0101-0200"
)
progress["updated_at"] = (
    datetime.now(timezone.utc).isoformat()
)

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== LONG PAUSE AUDIT SAVED ===")

print(
    audit[
        [
            "audio_id",
            "manual_pause_observation",
            "notes"
        ]
    ].to_string(index=False)
)

print("\nAudit rows:", len(audit))
print(
    "Pending observations:",
    (audit["manual_pause_observation"] == "pending").sum()
)
print("Checkpoint updated.")
print("Next task:", progress["next_task"])


In [ ]:

import io
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "features/acoustic/pilot_acoustic_features_0101-0200.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["configuration_name"] == "pilot_acoustic_v1"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot[
    pilot["pilot_index"].between(101, 200)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(101, 201))
assert batch["audio_id"].is_unique

expected_ids = set(batch["audio_id"])
feature_columns = config["features"]

if OUT.exists():
    result = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )
    print("Existing batch found; validating instead of recomputing.")

else:
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]
    model.eval()

    pf = pq.ParquetFile(TRAIN)
    audio_data = {}

    for group_id in range(pf.metadata.num_row_groups):
        audios = pf.read_row_group(
            group_id, columns=["audio"]
        ).column("audio").to_pylist()

        for item in audios:
            audio_id = Path(item["path"]).stem
            if audio_id in expected_ids:
                audio_data[audio_id] = item["bytes"]

        if len(audio_data) == 100:
            break

    assert set(audio_data) == expected_ids

    sc = config["silero_vad"]
    ec = config["energy_segmentation"]
    pc = config["pitch"]

    records = []

    with torch.no_grad():
        for n, (_, row) in enumerate(batch.iterrows(), start=1):

            audio_id = row["audio_id"]

            y, sr = sf.read(
                io.BytesIO(audio_data[audio_id]),
                dtype="float32"
            )

            assert sr == 16000 and y.ndim == 1
            duration = len(y) / sr

            vad = get_speech_timestamps(
                torch.from_numpy(y),
                model,
                sampling_rate=sr,
                threshold=sc["threshold"],
                min_speech_duration_ms=sc["min_speech_duration_ms"],
                min_silence_duration_ms=sc["min_silence_duration_ms"],
                speech_pad_ms=sc["speech_pad_ms"]
            )

            if vad:
                pad = int(sc["hybrid_padding_ms"] * sr / 1000)
                start = max(0, vad[0]["start"] - pad)
                end = min(len(y), vad[-1]["end"] + pad)

                intervals = librosa.effects.split(
                    y[start:end],
                    top_db=ec["top_db"],
                    frame_length=ec["frame_length"],
                    hop_length=ec["hop_length"]
                )

                intervals = [
                    (int(a + start), int(b + start))
                    for a, b in intervals
                    if (b-a)/sr >= ec["min_active_segment_ms"]/1000
                ]
            else:
                intervals = []

            speech_sec = sum(
                (b-a)/sr for a,b in intervals
            )

            gaps = [
                (intervals[i+1][0]-intervals[i][1])/sr
                for i in range(len(intervals)-1)
            ]

            pauses100 = [g for g in gaps if g >= 0.1]
            pauses200 = [g for g in gaps if g >= 0.2]

            f0, voiced_flag, _ = librosa.pyin(
                y,
                sr=sr,
                fmin=pc["fmin_hz"],
                fmax=pc["fmax_hz"],
                frame_length=pc["frame_length"],
                hop_length=pc["hop_length"],
                fill_na=np.nan
            )

            voiced_f0 = f0[
                voiced_flag & np.isfinite(f0)
            ]

            rms = librosa.feature.rms(
                y=y,
                frame_length=1024,
                hop_length=256
            )[0]

            records.append({
                "pilot_index": int(row["pilot_index"]),
                "audio_id": audio_id,
                "speaker": row["speaker"],
                "fluency": row["fluency"],
                "duration_sec": duration,
                "speech_duration_sec": speech_sec,
                "speech_ratio": speech_sec/duration,
                "pause_count_100ms": len(pauses100),
                "pause_count_200ms": len(pauses200),
                "pause_duration_200ms": sum(pauses200),
                "pause_ratio_200ms": sum(pauses200)/duration,
                "leading_silence_sec": (
                    intervals[0][0]/sr if intervals else np.nan
                ),
                "trailing_silence_sec": (
                    (len(y)-intervals[-1][1])/sr
                    if intervals else np.nan
                ),
                "pitch_mean_hz": (
                    float(np.mean(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "pitch_sd_hz": (
                    float(np.std(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "voiced_ratio": len(voiced_f0)/len(f0),
                "rms_mean": float(np.mean(rms)),
                "rms_sd": float(np.std(rms))
            })

            if n % 10 == 0:
                print(f"Processed {n}/100")

    result = pd.DataFrame(records)
    result.to_csv(OUT, index=False)

# Validation
assert len(result) == 100
assert result["pilot_index"].tolist() == list(range(101, 201))
assert result["audio_id"].is_unique
assert set(result["audio_id"]) == expected_ids
assert all(c in result.columns for c in feature_columns)

assert result["duration_sec"].gt(0).all()
assert result["speech_duration_sec"].ge(0).all()
assert (
    result["speech_duration_sec"]
    <= result["duration_sec"] + 1e-5
).all()
assert result["speech_ratio"].between(0, 1.00001).all()
assert result["voiced_ratio"].between(0, 1).all()

sha = hashlib.sha256(OUT.read_bytes()).hexdigest()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_pilot_processed_audio"] = 200
progress["step2_last_batch"] = "0101-0200"
progress["step2_last_batch_sha256"] = sha
progress["next_task"] = "Quality control for batch 0101-0200"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ACOUSTIC BATCH 0101-0200 ===")
print("Rows:", len(result))
print("Speakers:", result["speaker"].nunique())
print("Features:", len(feature_columns))
print("Missing pitch values:", result["pitch_mean_hz"].isna().sum())
print("Zero speech duration:",
      (result["speech_duration_sec"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\n=== FEATURE SUMMARY ===")
print(result[feature_columns].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", sha)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = PROJECT / "features/acoustic/pilot_acoustic_features_0101-0200.csv"
OUT = PROJECT / "reports/acoustic_qc_batch_0101-0200.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 100
assert df["pilot_index"].tolist() == list(range(101, 201))
assert df["audio_id"].is_unique

# Same QC thresholds as batch 1
df["flag_long_pause"] = df["pause_duration_200ms"] >= 1.5
df["flag_low_speech_ratio"] = df["speech_ratio"] < 0.4
df["flag_low_voiced_ratio"] = df["voiced_ratio"] < 0.2
df["flag_high_pitch_sd"] = df["pitch_sd_hz"] > 70

# Additional pitch extraction failure check
df["flag_missing_pitch"] = (
    df["pitch_mean_hz"].isna()
    | df["pitch_sd_hz"].isna()
)

df["flag_invalid_duration"] = (
    (df["duration_sec"] <= 0)
    | (df["speech_duration_sec"] < 0)
    | (df["speech_duration_sec"] > df["duration_sec"] + 1e-5)
)

df["flag_invalid_pause"] = (
    (df["pause_duration_200ms"] < 0)
    | (df["pause_duration_200ms"] > df["duration_sec"] + 1e-5)
    | (df["pause_count_200ms"] < 0)
    | (df["pause_count_200ms"] > df["pause_count_100ms"])
)

flag_cols = [c for c in df.columns if c.startswith("flag_")]
df["qc_flag_count"] = df[flag_cols].sum(axis=1)

flagged = df[df["qc_flag_count"] > 0].copy()
flagged = flagged.sort_values(
    ["qc_flag_count", "pause_duration_200ms"],
    ascending=[False, False]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
flagged.to_csv(OUT, index=False)

print("=== ACOUSTIC QC BATCH 0101-0200 ===")
print("Total recordings:", len(df))
print("Flagged recordings:", len(flagged))

print("\n=== FLAG COUNTS ===")
for col in flag_cols:
    print(col, ":", int(df[col].sum()))

print("\n=== FLAGGED RECORDINGS ===")
display_cols = [
    "pilot_index", "audio_id", "speaker", "fluency",
    "duration_sec", "speech_ratio",
    "pause_count_200ms", "pause_duration_200ms",
    "pitch_mean_hz", "pitch_sd_hz", "voiced_ratio",
    "qc_flag_count"
]

print(
    flagged[display_cols]
    .round(3)
    .to_string(index=False)
)

print("\n=== MISSING PITCH DETAILS ===")
print(
    df.loc[df["flag_missing_pitch"], display_cols]
    .round(3)
    .to_string(index=False)
)

print("\n=== TOP 5 LONGEST PAUSES ===")
print(
    df.nlargest(5, "pause_duration_200ms")[
        ["pilot_index", "audio_id", "speaker", "fluency",
         "duration_sec", "pause_count_200ms",
         "pause_duration_200ms"]
    ].round(3).to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_batch2_qc_completed"] = True
progress["step2_batch2_qc_flagged"] = len(flagged)
progress["next_task"] = "Review batch 2 pitch and segmentation flags"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nQC CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

OUT = PROJECT / "reports/pitch_failure_diagnostic_022420164.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

TARGET_ID = "022420164"

pf = pq.ParquetFile(TRAIN)
audio_bytes = None

for group_id in range(pf.metadata.num_row_groups):

    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:

        if Path(item["path"]).stem == TARGET_ID:
            audio_bytes = item["bytes"]
            break

    if audio_bytes is not None:
        break

assert audio_bytes is not None

y, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

assert sr == 16000
assert y.ndim == 1

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string"}
)

row = pilot.loc[
    pilot["audio_id"] == TARGET_ID
].iloc[0]

print("=== TARGET AUDIO ===")
print("Audio ID:", TARGET_ID)
print("Speaker:", row["speaker"])
print("Human fluency:", row["fluency"])
print("Expected text:", row["text"])
print("Duration:", round(len(y) / sr, 3))
print("Peak amplitude:", round(float(np.max(np.abs(y))), 5))

display(Audio(y, rate=sr, normalize=False))

rms = librosa.feature.rms(
    y=y,
    frame_length=1024,
    hop_length=256
)[0]

configs = [
    ("original_75_500", 75, 500),
    ("diagnostic_60_650", 60, 650)
]

records = []

for name, fmin, fmax in configs:

    f0, voiced_flag, voiced_prob = librosa.pyin(
        y,
        sr=sr,
        fmin=fmin,
        fmax=fmax,
        frame_length=1024,
        hop_length=256,
        fill_na=np.nan
    )

    valid = f0[
        voiced_flag & np.isfinite(f0)
    ]

    records.append({
        "audio_id": TARGET_ID,
        "configuration": name,
        "fmin_hz": fmin,
        "fmax_hz": fmax,
        "total_frames": len(f0),
        "voiced_frames": len(valid),
        "voiced_ratio": len(valid) / len(f0),
        "pitch_mean_hz": (
            float(np.mean(valid)) if len(valid) else np.nan
        ),
        "pitch_sd_hz": (
            float(np.std(valid)) if len(valid) else np.nan
        ),
        "mean_voiced_probability": (
            float(np.mean(voiced_prob[voiced_flag]))
            if np.any(voiced_flag) else np.nan
        ),
        "rms_mean": float(np.mean(rms)),
        "rms_max": float(np.max(rms))
    })

result = pd.DataFrame(records)
result.to_csv(OUT, index=False)

print("\n=== PITCH FAILURE DIAGNOSTICS ===")
print(result.round(4).to_string(index=False))

print("\nSaved:", OUT)


In [ ]:

import io
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

try:
    import parselmouth
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "praat-parselmouth"
    ])
    import parselmouth

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
OUT = PROJECT / "reports/praat_pitch_diagnostic_022420164.csv"

TARGET_ID = "022420164"
OUT.parent.mkdir(parents=True, exist_ok=True)

pf = pq.ParquetFile(TRAIN)
audio_bytes = None

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        if Path(audio["path"]).stem == TARGET_ID:
            audio_bytes = audio["bytes"]
            break

    if audio_bytes is not None:
        break

assert audio_bytes is not None

y, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float64"
)

assert sr == 16000 and y.ndim == 1

sound = parselmouth.Sound(y, sampling_frequency=sr)

configs = [
    ("praat_75_500", 75, 500),
    ("praat_60_650", 60, 650)
]

records = []

for name, floor, ceiling in configs:

    pitch = sound.to_pitch_ac(
        time_step=0.016,
        pitch_floor=floor,
        pitch_ceiling=ceiling
    )

    frequencies = pitch.selected_array["frequency"]

    voiced = frequencies[
        np.isfinite(frequencies) & (frequencies > 0)
    ]

    records.append({
        "audio_id": TARGET_ID,
        "method": name,
        "pitch_floor_hz": floor,
        "pitch_ceiling_hz": ceiling,
        "total_frames": len(frequencies),
        "voiced_frames": len(voiced),
        "voiced_ratio": (
            len(voiced) / len(frequencies)
            if len(frequencies) else np.nan
        ),
        "pitch_mean_hz": (
            float(np.mean(voiced))
            if len(voiced) else np.nan
        ),
        "pitch_sd_hz": (
            float(np.std(voiced))
            if len(voiced) else np.nan
        ),
        "pitch_min_hz": (
            float(np.min(voiced))
            if len(voiced) else np.nan
        ),
        "pitch_max_hz": (
            float(np.max(voiced))
            if len(voiced) else np.nan
        )
    })

result = pd.DataFrame(records)
result.to_csv(OUT, index=False)

print("=== PRAAT PITCH DIAGNOSTICS ===")
print("Parselmouth version:", parselmouth.__version__)
print(result.round(3).to_string(index=False))

print("\nSaved:", OUT)
print("Original acoustic batch CSV files unchanged.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

PYIN = PROJECT / "reports/pitch_failure_diagnostic_022420164.csv"
PRAAT = PROJECT / "reports/praat_pitch_diagnostic_022420164.csv"
REPORT = PROJECT / "reports/step02_pitch_failure_review.md"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert PYIN.exists()
assert PRAAT.exists()
assert CHECK.exists()

pyin = pd.read_csv(PYIN)
praat = pd.read_csv(PRAAT)

assert len(pyin) == 2
assert len(praat) == 2

report = """# STEP 2 - Pitch Extraction Failure Review

## Recording
Audio ID: 022420164
Speaker: 2242
Human fluency: 10

## Researcher listening observation
Speech was clearly audible and pronunciation sounded
very good. Background noise or room reverberation
was reported.

## pYIN diagnostics
Original F0 range: 75-500 Hz.
Expanded F0 range: 60-650 Hz.
Neither configuration detected voiced frames.

## Praat diagnostics
Praat autocorrelation detected voiced frames.

75-500 Hz:
- 43/199 voiced frames
- Mean F0: 139.285 Hz
- F0 SD: 32.528 Hz

60-650 Hz:
- 42/199 voiced frames
- Mean F0: 138.502 Hz
- F0 SD: 34.499 Hz

## Interpretation
This is an estimator-specific pitch extraction discrepancy.
The exact cause has not been established.
Praat values are diagnostic, not manually verified
ground-truth pitch estimates.

## Methodological decision
- Preserve original pYIN NaN values.
- Do not replace them with Praat measurements.
- Do not drop the recording.
- Keep pilot_acoustic_v1 unchanged.
- Handle missing pitch values within training folds.
- Consider a uniform Praat-based sensitivity analysis later.

## Status
Diagnostics completed; pilot extraction may continue.
"""

REPORT.write_text(report, encoding="utf-8")

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step2_pitch_failure_audio_id"] = "022420164"
progress["step2_pitch_failure_review"] = "completed"
progress["step2_pitch_missing_policy"] = "preserve_nan"
progress["step2_pilot_processed_audio"] = 200
progress["next_task"] = (
    "Extract acoustic features for pilot rows 0201-0300"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== PITCH FAILURE REVIEW SAVED ===")
print("Audio:", progress["step2_pitch_failure_audio_id"])
print("pYIN diagnostics:", len(pyin))
print("Praat diagnostics:", len(praat))
print("Missing pitch policy:", progress["step2_pitch_missing_policy"])
print("Pilot processed:", progress["step2_pilot_processed_audio"])
print("Report:", REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"
OUT = PROJECT / "features/acoustic/pilot_acoustic_features_0201-0300.csv"

OUT.parent.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["configuration_name"] == "pilot_acoustic_v1"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot[
    pilot["pilot_index"].between(201, 300)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(201, 301))
assert batch["audio_id"].is_unique

expected_ids = set(batch["audio_id"])
feature_columns = config["features"]

if OUT.exists():
    result = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )
    print("Existing batch found; validating instead of recomputing.")

else:
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]
    model.eval()

    pf = pq.ParquetFile(TRAIN)
    audio_data = {}

    for group_id in range(pf.metadata.num_row_groups):
        audios = pf.read_row_group(
            group_id, columns=["audio"]
        ).column("audio").to_pylist()

        for item in audios:
            audio_id = Path(item["path"]).stem
            if audio_id in expected_ids:
                audio_data[audio_id] = item["bytes"]

        if len(audio_data) == 100:
            break

    assert set(audio_data) == expected_ids

    sc = config["silero_vad"]
    ec = config["energy_segmentation"]
    pc = config["pitch"]
    records = []

    with torch.no_grad():
        for n, (_, row) in enumerate(batch.iterrows(), start=1):

            audio_id = row["audio_id"]
            y, sr = sf.read(
                io.BytesIO(audio_data[audio_id]),
                dtype="float32"
            )

            assert sr == 16000 and y.ndim == 1
            duration = len(y) / sr

            vad = get_speech_timestamps(
                torch.from_numpy(y),
                model,
                sampling_rate=sr,
                threshold=sc["threshold"],
                min_speech_duration_ms=sc["min_speech_duration_ms"],
                min_silence_duration_ms=sc["min_silence_duration_ms"],
                speech_pad_ms=sc["speech_pad_ms"]
            )

            if vad:
                pad = int(sc["hybrid_padding_ms"] * sr / 1000)
                start = max(0, vad[0]["start"] - pad)
                end = min(len(y), vad[-1]["end"] + pad)

                intervals = librosa.effects.split(
                    y[start:end],
                    top_db=ec["top_db"],
                    frame_length=ec["frame_length"],
                    hop_length=ec["hop_length"]
                )

                intervals = [
                    (int(a + start), int(b + start))
                    for a, b in intervals
                    if (b-a)/sr >= ec["min_active_segment_ms"]/1000
                ]
            else:
                intervals = []

            speech_sec = sum(
                (b-a)/sr for a, b in intervals
            )

            gaps = [
                (intervals[i+1][0]-intervals[i][1])/sr
                for i in range(len(intervals)-1)
            ]

            pauses100 = [g for g in gaps if g >= 0.1]
            pauses200 = [g for g in gaps if g >= 0.2]

            f0, voiced_flag, _ = librosa.pyin(
                y,
                sr=sr,
                fmin=pc["fmin_hz"],
                fmax=pc["fmax_hz"],
                frame_length=pc["frame_length"],
                hop_length=pc["hop_length"],
                fill_na=np.nan
            )

            voiced_f0 = f0[
                voiced_flag & np.isfinite(f0)
            ]

            rms = librosa.feature.rms(
                y=y,
                frame_length=1024,
                hop_length=256
            )[0]

            records.append({
                "pilot_index": int(row["pilot_index"]),
                "audio_id": audio_id,
                "speaker": row["speaker"],
                "fluency": row["fluency"],
                "duration_sec": duration,
                "speech_duration_sec": speech_sec,
                "speech_ratio": speech_sec / duration,
                "pause_count_100ms": len(pauses100),
                "pause_count_200ms": len(pauses200),
                "pause_duration_200ms": sum(pauses200),
                "pause_ratio_200ms": sum(pauses200)/duration,
                "leading_silence_sec": (
                    intervals[0][0]/sr if intervals else np.nan
                ),
                "trailing_silence_sec": (
                    (len(y)-intervals[-1][1])/sr
                    if intervals else np.nan
                ),
                "pitch_mean_hz": (
                    float(np.mean(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "pitch_sd_hz": (
                    float(np.std(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "voiced_ratio": len(voiced_f0)/len(f0),
                "rms_mean": float(np.mean(rms)),
                "rms_sd": float(np.std(rms))
            })

            if n % 10 == 0:
                print(f"Processed {n}/100")

    result = pd.DataFrame(records)
    result.to_csv(OUT, index=False)

# Full validation
assert len(result) == 100
assert result["pilot_index"].tolist() == list(range(201, 301))
assert result["audio_id"].is_unique
assert set(result["audio_id"]) == expected_ids
assert all(c in result.columns for c in feature_columns)

assert result["duration_sec"].gt(0).all()
assert result["speech_duration_sec"].ge(0).all()
assert (
    result["speech_duration_sec"]
    <= result["duration_sec"] + 1e-5
).all()
assert result["speech_ratio"].between(0, 1.00001).all()
assert result["voiced_ratio"].between(0, 1).all()

sha = hashlib.sha256(OUT.read_bytes()).hexdigest()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_pilot_processed_audio"] = 300
progress["step2_last_batch"] = "0201-0300"
progress["step2_last_batch_sha256"] = sha
progress["next_task"] = "Quality control for batch 0201-0300"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ACOUSTIC BATCH 0201-0300 ===")
print("Rows:", len(result))
print("Speakers:", result["speaker"].nunique())
print("Features:", len(feature_columns))
print("Missing pitch values:", result["pitch_mean_hz"].isna().sum())
print("Zero speech duration:",
      (result["speech_duration_sec"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\n=== FEATURE SUMMARY ===")
print(result[feature_columns].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", sha)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = PROJECT / "features/acoustic/pilot_acoustic_features_0201-0300.csv"
OUT = PROJECT / "reports/acoustic_qc_batch_0201-0300.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 100
assert df["pilot_index"].tolist() == list(range(201, 301))
assert df["audio_id"].is_unique

# Original QC thresholds
df["flag_long_pause"] = df["pause_duration_200ms"] >= 1.5
df["flag_low_speech_ratio"] = df["speech_ratio"] < 0.4
df["flag_low_voiced_ratio"] = df["voiced_ratio"] < 0.2
df["flag_high_pitch_sd"] = df["pitch_sd_hz"] > 70
df["flag_missing_pitch"] = (
    df["pitch_mean_hz"].isna()
    | df["pitch_sd_hz"].isna()
)

# Additional diagnostic check for batch 3
df["flag_long_trailing_silence"] = (
    df["trailing_silence_sec"] >= 2.0
)

# Numerical consistency
df["flag_invalid_duration"] = (
    (df["duration_sec"] <= 0)
    | (df["speech_duration_sec"] < 0)
    | (df["speech_duration_sec"] > df["duration_sec"] + 1e-5)
)

df["flag_invalid_pause"] = (
    (df["pause_duration_200ms"] < 0)
    | (df["pause_duration_200ms"] > df["duration_sec"] + 1e-5)
    | (df["pause_count_200ms"] < 0)
    | (df["pause_count_200ms"] > df["pause_count_100ms"])
)

flag_cols = [c for c in df.columns if c.startswith("flag_")]
df["qc_flag_count"] = df[flag_cols].sum(axis=1)

flagged = df[df["qc_flag_count"] > 0].copy()
flagged = flagged.sort_values(
    ["qc_flag_count", "trailing_silence_sec"],
    ascending=[False, False]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
flagged.to_csv(OUT, index=False)

print("=== ACOUSTIC QC BATCH 0201-0300 ===")
print("Total recordings:", len(df))
print("Flagged recordings:", len(flagged))

print("\n=== FLAG COUNTS ===")
for col in flag_cols:
    print(col, ":", int(df[col].sum()))

display_cols = [
    "pilot_index", "audio_id", "speaker", "fluency",
    "duration_sec", "speech_ratio",
    "pause_count_200ms", "pause_duration_200ms",
    "trailing_silence_sec", "pitch_mean_hz",
    "pitch_sd_hz", "voiced_ratio", "qc_flag_count"
]

print("\n=== FLAGGED RECORDINGS ===")
print(
    flagged[display_cols]
    .round(3)
    .to_string(index=False)
)

print("\n=== TOP 5 TRAILING SILENCE ===")
print(
    df.nlargest(5, "trailing_silence_sec")[
        [
            "pilot_index", "audio_id", "speaker",
            "fluency", "duration_sec",
            "speech_duration_sec", "speech_ratio",
            "trailing_silence_sec"
        ]
    ].round(3).to_string(index=False)
)

print("\n=== MISSING PITCH DETAILS ===")
print(
    df.loc[df["flag_missing_pitch"], display_cols]
    .round(3)
    .to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_batch3_qc_completed"] = True
progress["step2_batch3_qc_flagged"] = len(flagged)
progress["next_task"] = "Review third-batch QC anomalies"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nQC CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import matplotlib.pyplot as plt
import librosa

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
FEATURES = PROJECT / "features/acoustic/pilot_acoustic_features_0201-0300.csv"

FIG_DIR = PROJECT / "results/figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

target_ids = ["060100124", "069120025"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"}
)

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == set(target_ids)

print("=== CRITICAL AUDIO INSPECTION ===")

for audio_id in target_ids:

    row = pilot.loc[
        pilot["audio_id"] == audio_id
    ].iloc[0]

    feat = features.loc[
        features["audio_id"] == audio_id
    ].iloc[0]

    y, sr = audio_data[audio_id]
    duration = len(y) / sr

    print("\n" + "=" * 55)
    print("Audio ID:", audio_id)
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(duration, 3))
    print("Speech ratio:", round(feat["speech_ratio"], 3))
    print("Trailing silence:", round(
        feat["trailing_silence_sec"], 3
    ))
    print("Voiced ratio:", round(feat["voiced_ratio"], 3))

    display(Audio(y, rate=sr, normalize=False))

    rms = librosa.feature.rms(
        y=y,
        frame_length=512,
        hop_length=160
    )[0]

    rms_times = librosa.frames_to_time(
        np.arange(len(rms)),
        sr=sr,
        hop_length=160
    )

    fig, ax = plt.subplots(figsize=(12, 3))

    times = np.arange(len(y)) / sr
    ax.plot(times, y, linewidth=0.5, alpha=0.7)
    ax.set_title(f"Waveform: {audio_id}")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, duration)

    ax2 = ax.twinx()
    ax2.plot(rms_times, rms, linewidth=1.1, alpha=0.8)
    ax2.set_ylabel("RMS energy")

    fig.tight_layout()

    fig_path = FIG_DIR / f"qc_waveform_{audio_id}.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    plt.close(fig)

    print("Figure saved:", fig_path)

print("\nInspection finished.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

OUT = PROJECT / "data/metadata/critical_qc_audit_batch3.csv"
REPORT = PROJECT / "reports/step02_batch3_critical_qc_review.md"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT.parent.mkdir(parents=True, exist_ok=True)

records = [
    {
        "audio_id": "060100124",
        "pilot_index": 254,
        "fluency": 8,
        "anomaly": "long_trailing_silence",
        "researcher_observation": (
            "Long silence audible after AS I COULD"
        ),
        "waveform_observation": (
            "Main speech concentrated near beginning; "
            "later low-energy region with isolated impulses"
        ),
        "decision": "retain_original_feature_values",
        "manual_timestamp_validation": False
    },
    {
        "audio_id": "069120025",
        "pilot_index": 264,
        "fluency": 7,
        "anomaly": "pyin_pitch_failure",
        "researcher_observation": (
            "Two throat-clearing-like sounds after speech"
        ),
        "waveform_observation": (
            "Strong signal activity near 2.2-2.6 seconds; "
            "source of signals not independently confirmed"
        ),
        "decision": "preserve_pitch_nan",
        "manual_timestamp_validation": False
    }
]

audit = pd.DataFrame(records)

if OUT.exists():
    old = pd.read_csv(OUT, dtype={"audio_id": "string"})
    assert set(old["audio_id"]) == set(audit["audio_id"])

audit.to_csv(OUT, index=False, encoding="utf-8")

report = """# STEP 2 - Batch 3 Critical QC Review

## Dataset
Speechocean762 - train pilot.

## Recording 060100124
Human fluency: 8
Duration: 5.62 seconds
Hybrid speech duration: 1.00 seconds
Trailing silence: 4.50 seconds

Researcher reported long silence after the spoken phrase.
Waveform shows early speech activity followed by mostly
low-energy signal and isolated impulses.

Decision:
Retain original measurements.
No evidence sufficient to automatically relabel
this as a segmentation error.

## Recording 069120025
Human fluency: 7
pYIN voiced ratio: 0
pYIN pitch mean and SD: NaN

Researcher reported two throat-clearing-like sounds
after the phrase. Waveform shows strong later activity.

Decision:
Preserve NaN pitch values.
Do not substitute another estimator without applying
the same method consistently across recordings.

## Limitations
Observations are qualitative.
No manually timestamped reference segmentation.
No definitive cause of the pYIN failure established.

## Next
Continue pilot acoustic extraction with unchanged
pilot_acoustic_v1 configuration.
"""

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(report, encoding="utf-8")

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step2_batch3_critical_audit_count"] = 2
progress["step2_batch3_critical_review"] = "completed"
progress["step2_pilot_processed_audio"] = 300
progress["next_task"] = (
    "Extract acoustic features for pilot rows 0301-0400"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== BATCH 3 CRITICAL AUDIT SAVED ===")
print(audit[
    ["audio_id", "anomaly", "decision"]
].to_string(index=False))

print("\nAudit rows:", len(audit))
print("Pilot processed:",
      progress["step2_pilot_processed_audio"])
print("Report:", REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "features/acoustic/pilot_acoustic_features_0301-0400.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["configuration_name"] == "pilot_acoustic_v1"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot[
    pilot["pilot_index"].between(301, 400)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(301, 401))
assert batch["audio_id"].is_unique

expected_ids = set(batch["audio_id"])
feature_columns = config["features"]

if OUT.exists():
    result = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )
    print("Existing batch found; validating instead of recomputing.")

else:
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]
    model.eval()

    pf = pq.ParquetFile(TRAIN)
    audio_data = {}

    for group_id in range(pf.metadata.num_row_groups):
        audios = pf.read_row_group(
            group_id, columns=["audio"]
        ).column("audio").to_pylist()

        for item in audios:
            audio_id = Path(item["path"]).stem
            if audio_id in expected_ids:
                audio_data[audio_id] = item["bytes"]

        if len(audio_data) == 100:
            break

    assert set(audio_data) == expected_ids

    sc = config["silero_vad"]
    ec = config["energy_segmentation"]
    pc = config["pitch"]
    records = []

    with torch.no_grad():
        for n, (_, row) in enumerate(batch.iterrows(), start=1):
            audio_id = row["audio_id"]

            y, sr = sf.read(
                io.BytesIO(audio_data[audio_id]),
                dtype="float32"
            )

            assert sr == config["sample_rate_hz"]
            assert y.ndim == 1
            duration = len(y) / sr

            vad = get_speech_timestamps(
                torch.from_numpy(y),
                model,
                sampling_rate=sr,
                threshold=sc["threshold"],
                min_speech_duration_ms=sc["min_speech_duration_ms"],
                min_silence_duration_ms=sc["min_silence_duration_ms"],
                speech_pad_ms=sc["speech_pad_ms"]
            )

            if vad:
                pad = int(sc["hybrid_padding_ms"] * sr / 1000)
                start = max(0, vad[0]["start"] - pad)
                end = min(len(y), vad[-1]["end"] + pad)

                intervals = librosa.effects.split(
                    y[start:end],
                    top_db=ec["top_db"],
                    frame_length=ec["frame_length"],
                    hop_length=ec["hop_length"]
                )

                intervals = [
                    (int(a + start), int(b + start))
                    for a, b in intervals
                    if (b-a)/sr >= ec["min_active_segment_ms"]/1000
                ]
            else:
                intervals = []

            speech_sec = sum(
                (b-a)/sr for a, b in intervals
            )

            gaps = [
                (intervals[i+1][0]-intervals[i][1])/sr
                for i in range(len(intervals)-1)
            ]

            pauses100 = [g for g in gaps if g >= 0.1]
            pauses200 = [g for g in gaps if g >= 0.2]

            f0, voiced_flag, _ = librosa.pyin(
                y,
                sr=sr,
                fmin=pc["fmin_hz"],
                fmax=pc["fmax_hz"],
                frame_length=pc["frame_length"],
                hop_length=pc["hop_length"],
                fill_na=np.nan
            )

            voiced_f0 = f0[
                voiced_flag & np.isfinite(f0)
            ]

            rms = librosa.feature.rms(
                y=y,
                frame_length=1024,
                hop_length=256
            )[0]

            records.append({
                "pilot_index": int(row["pilot_index"]),
                "audio_id": audio_id,
                "speaker": row["speaker"],
                "fluency": row["fluency"],
                "duration_sec": duration,
                "speech_duration_sec": speech_sec,
                "speech_ratio": speech_sec / duration,
                "pause_count_100ms": len(pauses100),
                "pause_count_200ms": len(pauses200),
                "pause_duration_200ms": sum(pauses200),
                "pause_ratio_200ms": sum(pauses200)/duration,
                "leading_silence_sec": (
                    intervals[0][0]/sr if intervals else np.nan
                ),
                "trailing_silence_sec": (
                    (len(y)-intervals[-1][1])/sr
                    if intervals else np.nan
                ),
                "pitch_mean_hz": (
                    float(np.mean(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "pitch_sd_hz": (
                    float(np.std(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "voiced_ratio": len(voiced_f0)/len(f0),
                "rms_mean": float(np.mean(rms)),
                "rms_sd": float(np.std(rms))
            })

            if n % 10 == 0:
                print(f"Processed {n}/100")

    result = pd.DataFrame(records)
    assert len(result) == 100
    result.to_csv(OUT, index=False)

# Validation against original pilot metadata
assert len(result) == 100
assert result["pilot_index"].tolist() == list(range(301, 401))
assert result["audio_id"].is_unique
assert set(result["audio_id"]) == expected_ids
assert all(c in result.columns for c in feature_columns)

reference = batch.set_index("pilot_index")
observed = result.set_index("pilot_index")
assert (
    observed["audio_id"].tolist()
    == reference["audio_id"].tolist()
)
assert (
    observed["speaker"].astype(str).tolist()
    == reference["speaker"].astype(str).tolist()
)

assert result["duration_sec"].gt(0).all()
assert result["speech_duration_sec"].ge(0).all()
assert (
    result["speech_duration_sec"]
    <= result["duration_sec"] + 1e-5
).all()
assert result["speech_ratio"].between(0, 1.00001).all()
assert result["voiced_ratio"].between(0, 1).all()
assert result["pause_count_200ms"].le(
    result["pause_count_100ms"]
).all()

sha = hashlib.sha256(OUT.read_bytes()).hexdigest()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_pilot_processed_audio"] = 400
progress["step2_last_batch"] = "0301-0400"
progress["step2_last_batch_sha256"] = sha
progress["step2_batch4_extraction_completed"] = True
progress["next_task"] = "Quality control for batch 0301-0400"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ACOUSTIC BATCH 0301-0400 ===")
print("Rows:", len(result))
print("Speakers:", result["speaker"].nunique())
print("Features:", len(feature_columns))
print("Missing pitch values:", result["pitch_mean_hz"].isna().sum())
print("Zero speech duration:",
      (result["speech_duration_sec"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\n=== FEATURE SUMMARY ===")
print(result[feature_columns].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", sha)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "features/acoustic/pilot_acoustic_features_0301-0400.csv"
OUT = PROJECT / "reports/acoustic_qc_batch_0301-0400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 100
assert df["pilot_index"].tolist() == list(range(301, 401))
assert df["audio_id"].is_unique

# Same QC thresholds as previous batches
df["flag_long_pause"] = df["pause_duration_200ms"] >= 1.5
df["flag_low_speech_ratio"] = df["speech_ratio"] < 0.4
df["flag_low_voiced_ratio"] = df["voiced_ratio"] < 0.2
df["flag_high_pitch_sd"] = df["pitch_sd_hz"] > 70
df["flag_missing_pitch"] = (
    df["pitch_mean_hz"].isna()
    | df["pitch_sd_hz"].isna()
)
df["flag_long_trailing_silence"] = (
    df["trailing_silence_sec"] >= 2.0
)

df["flag_invalid_duration"] = (
    (df["duration_sec"] <= 0)
    | (df["speech_duration_sec"] < 0)
    | (df["speech_duration_sec"] > df["duration_sec"] + 1e-5)
)

df["flag_invalid_pause"] = (
    (df["pause_duration_200ms"] < 0)
    | (df["pause_duration_200ms"] > df["duration_sec"] + 1e-5)
    | (df["pause_count_200ms"] < 0)
    | (df["pause_count_200ms"] > df["pause_count_100ms"])
)

flag_cols = [c for c in df.columns if c.startswith("flag_")]
df["qc_flag_count"] = df[flag_cols].sum(axis=1)

flagged = df[df["qc_flag_count"] > 0].copy()
flagged = flagged.sort_values(
    ["qc_flag_count", "pause_duration_200ms"],
    ascending=[False, False]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
flagged.to_csv(OUT, index=False)

print("=== ACOUSTIC QC BATCH 0301-0400 ===")
print("Total recordings:", len(df))
print("Flagged recordings:", len(flagged))

print("\n=== FLAG COUNTS ===")
for col in flag_cols:
    print(col, ":", int(df[col].sum()))

display_cols = [
    "pilot_index", "audio_id", "speaker", "fluency",
    "duration_sec", "speech_ratio",
    "pause_count_200ms", "pause_duration_200ms",
    "trailing_silence_sec", "pitch_mean_hz",
    "pitch_sd_hz", "voiced_ratio", "qc_flag_count"
]

print("\n=== TOP 20 FLAGGED RECORDINGS ===")
print(
    flagged[display_cols]
    .head(20)
    .round(3)
    .to_string(index=False)
)

print("\n=== TOP 5 LONGEST PAUSES ===")
print(
    df.nlargest(5, "pause_duration_200ms")[
        [
            "pilot_index", "audio_id", "speaker",
            "fluency", "duration_sec",
            "pause_count_200ms", "pause_duration_200ms"
        ]
    ].round(3).to_string(index=False)
)

print("\n=== TOP 5 PITCH SD ===")
print(
    df.nlargest(5, "pitch_sd_hz")[
        ["pilot_index", "audio_id", "speaker",
         "fluency", "pitch_mean_hz",
         "pitch_sd_hz", "voiced_ratio"]
    ].round(3).to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_batch4_qc_completed"] = True
progress["step2_batch4_qc_flagged"] = len(flagged)
progress["next_task"] = "Review batch 4 anomalies and merge pilot features"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nQC CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
FEATURES = PROJECT / "features/acoustic/pilot_acoustic_features_0301-0400.csv"

OUT = PROJECT / "data/metadata/batch4_long_pause_audit_3.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

target_ids = [
    "096340020",
    "096340017",
    "096140003"
]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"}
)

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):

    audios = pf.read_row_group(
        group_id,
        columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == set(target_ids)

records = []

print("=== BATCH 4 LONG PAUSE LISTENING ===")

for audio_id in target_ids:

    row = pilot.loc[
        pilot["audio_id"] == audio_id
    ].iloc[0]

    feat = features.loc[
        features["audio_id"] == audio_id
    ].iloc[0]

    y, sr = audio_data[audio_id]

    print("\n" + "=" * 55)
    print("Audio ID:", audio_id)
    print("Speaker:", row["speaker"])
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(len(y) / sr, 3))
    print("Hybrid pause count:",
          int(feat["pause_count_200ms"]))
    print("Hybrid pause duration:",
          round(feat["pause_duration_200ms"], 3))

    display(Audio(y, rate=sr, normalize=False))

    records.append({
        "audio_id": audio_id,
        "speaker": row["speaker"],
        "fluency": row["fluency"],
        "hybrid_pause_count_200ms": int(
            feat["pause_count_200ms"]
        ),
        "hybrid_pause_duration_200ms": float(
            feat["pause_duration_200ms"]
        ),
        "manual_pause_observation": "pending",
        "pronunciation_observation": "pending",
        "notes": ""
    })

audit = pd.DataFrame(records)

for col in [
    "manual_pause_observation",
    "pronunciation_observation",
    "notes"
]:
    audit[col] = audit[col].astype("string")

audit.to_csv(OUT, index=False)

print("\n=== AUDIT SUMMARY ===")
print("Audio players:", len(audio_data))
print("Audit rows:", len(audit))
print("Saved:", OUT)


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

AUDIT = PROJECT / "data/metadata/batch4_long_pause_audit_3.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Listening audit CSV not found"
assert CHECK.exists(), "Research checkpoint not found"

audit = pd.read_csv(
    AUDIT,
    dtype={"audio_id": "string", "speaker": "string"}
)

annotations = {
    "096340020": {
        "manual_pause_observation": "many_pauses_audible",
        "pronunciation_observation": "slow_hesitant_speech",
        "notes": (
            "Many internal pauses audible. "
            "Speech is slow and hesitant."
        )
    },
    "096340017": {
        "manual_pause_observation": "many_pauses_audible",
        "pronunciation_observation": "slow_hesitant_speech",
        "notes": (
            "Many internal pauses audible. "
            "Speech is slow and hesitant."
        )
    },
    "096140003": {
        "manual_pause_observation": "many_pauses_audible",
        "pronunciation_observation": "slow_hesitant_speech_with_repetition",
        "notes": (
            "Many internal pauses audible. "
            "Speech is slow and hesitant. "
            "The word STRANGE was repeated twice."
        )
    }
}

assert len(audit) == 3
assert audit["audio_id"].is_unique
assert set(audit["audio_id"]) == set(annotations)

for column in [
    "manual_pause_observation",
    "pronunciation_observation",
    "notes"
]:
    audit[column] = audit[column].astype("string")

for audio_id, values in annotations.items():
    mask = audit["audio_id"] == audio_id

    for column, value in values.items():
        audit.loc[mask, column] = value

audit["audit_type"] = "informal_researcher_listening"
audit["manual_pause_timestamps_available"] = False
audit["pause_duration_manually_verified"] = False

audit.to_csv(
    AUDIT,
    index=False,
    encoding="utf-8"
)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step2_batch4_listening_audit_count"] = 3
progress["step2_batch4_listening_audit_status"] = "completed"
progress["step2_batch4_qc_review_status"] = "preliminary_pass"
progress["step2_pilot_processed_audio"] = 400
progress["next_task"] = (
    "Merge four acoustic feature batches and validate 400 recordings"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== BATCH 4 LISTENING AUDIT SAVED ===")
print(
    audit[
        [
            "audio_id",
            "speaker",
            "manual_pause_observation",
            "pronunciation_observation",
            "notes"
        ]
    ].to_string(index=False)
)

print("\nAudit rows:", len(audit))
print(
    "Pending observations:",
    (
        audit["manual_pause_observation"] == "pending"
    ).sum()
)
print("Pilot processed:", progress["step2_pilot_processed_audio"])
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

FEATURE_DIR = PROJECT / "features/acoustic"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = FEATURE_DIR / "pilot_acoustic_features_400.csv"
REPORT = PROJECT / "reports/step02_merged_validation.json"

ranges = [
    (1, 100),
    (101, 200),
    (201, 300),
    (301, 400)
]

config = json.loads(CONFIG.read_text(encoding="utf-8"))
features = config["features"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("pilot_index").reset_index(drop=True)

assert len(pilot) == 400
assert pilot["pilot_index"].tolist() == list(range(1, 401))
assert pilot["audio_id"].is_unique

batches = []
batch_hashes = {}

for start, end in ranges:
    filename = (
        f"pilot_acoustic_features_{start:04d}-{end:04d}.csv"
    )
    path = FEATURE_DIR / filename

    assert path.exists(), f"Missing batch: {filename}"

    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

    assert len(df) == 100, filename
    assert df["pilot_index"].tolist() == list(
        range(start, end + 1)
    ), filename

    assert df["audio_id"].is_unique, filename
    assert all(c in df.columns for c in features), filename

    batch_hashes[filename] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    batches.append(df)

merged = pd.concat(batches, ignore_index=True)
merged = merged.sort_values(
    "pilot_index"
).reset_index(drop=True)

# Record-level identity checks
assert len(merged) == 400
assert merged["pilot_index"].tolist() == list(range(1, 401))
assert merged["audio_id"].is_unique

assert merged["audio_id"].tolist() == pilot["audio_id"].tolist()
assert merged["speaker"].tolist() == pilot["speaker"].tolist()
assert np.allclose(
    merged["fluency"].astype(float),
    pilot["fluency"].astype(float)
)

# Speaker validation
speaker_counts = merged.groupby("speaker").size()

assert merged["speaker"].nunique() == 20
assert (speaker_counts == 20).all()

# Feature validation
assert all(c in merged.columns for c in features)
assert merged["duration_sec"].gt(0).all()

assert merged["speech_duration_sec"].ge(0).all()
assert (
    merged["speech_duration_sec"]
    <= merged["duration_sec"] + 1e-5
).all()

assert merged["speech_ratio"].between(0, 1.00001).all()
assert merged["pause_ratio_200ms"].between(0, 1.00001).all()
assert merged["voiced_ratio"].between(0, 1).all()

assert (
    merged["pause_count_200ms"]
    <= merged["pause_count_100ms"]
).all()

# Validate pitch missingness consistently
missing_pitch = merged["pitch_mean_hz"].isna()

assert missing_pitch.equals(
    merged["pitch_sd_hz"].isna()
)

assert (
    (merged.loc[missing_pitch, "voiced_ratio"] == 0)
).all()

# Metadata completeness
assert merged["audio_id"].notna().all()
assert merged["speaker"].notna().all()
assert merged["fluency"].notna().all()

# Keep the original feature values intact
merged.to_csv(OUT, index=False)

merged_hash = hashlib.sha256(
    OUT.read_bytes()
).hexdigest()

validation = {
    "dataset": "Speechocean762",
    "configuration": config["configuration_name"],
    "rows": int(len(merged)),
    "unique_audio_ids": int(merged["audio_id"].nunique()),
    "unique_speakers": int(merged["speaker"].nunique()),
    "features": len(features),
    "missing_pitch_count": int(missing_pitch.sum()),
    "missing_pitch_audio_ids": merged.loc[
        missing_pitch, "audio_id"
    ].tolist(),
    "zero_speech_duration": int(
        (merged["speech_duration_sec"] == 0).sum()
    ),
    "audios_with_200ms_pauses": int(
        (merged["pause_count_200ms"] > 0).sum()
    ),
    "batch_sha256": batch_hashes,
    "merged_sha256": merged_hash,
    "status": "PASS",
    "validated_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(validation, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_merged_audio_count"] = 400
progress["step2_merged_validation"] = "PASS"
progress["step2_merged_sha256"] = merged_hash
progress["next_task"] = "Prepare final Step 2 report and backup"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 2 MERGED VALIDATION ===")
print("Rows:", validation["rows"])
print("Unique audio IDs:", validation["unique_audio_ids"])
print("Speakers:", validation["unique_speakers"])
print("Features:", validation["features"])
print("Missing pitch:", validation["missing_pitch_count"])
print("Missing pitch IDs:", validation["missing_pitch_audio_ids"])
print(
    "Audios with 200ms pauses:",
    validation["audios_with_200ms_pauses"]
)
print("Zero speech duration:", validation["zero_speech_duration"])
print("Validation:", validation["status"])

print("\n=== MERGED FEATURE SUMMARY ===")
print(merged[features].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", merged_hash)
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
FEATURES = PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
VALIDATION = PROJECT / "reports/step02_merged_validation.json"
CHECK = PROJECT / "checkpoints/research_progress.json"
REPORT = PROJECT / "reports/step02_final_report.md"

ZIP = Path("/kaggle/working/asr_fluency_step02_complete.zip")

df = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"}
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
validation = json.loads(VALIDATION.read_text(encoding="utf-8"))
progress = json.loads(CHECK.read_text(encoding="utf-8"))

assert len(df) == 400
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert validation["status"] == "PASS"
assert validation["rows"] == 400
assert len(config["features"]) == 14

missing_ids = df.loc[
    df["pitch_mean_hz"].isna(), "audio_id"
].tolist()

assert set(missing_ids) == {
    "022420164", "069120025"
}

# Confirm that no analysis or training has been performed
report = f"""# STEP 2 - Final Acoustic Feature Extraction Report

## Research
Within-Utterance ASR Transcription Variability and the
Stability of Human-Aligned L2 Fluency Scoring:
An Acoustic-Linguistic Comparison

## Dataset
Speechocean762
Official train split only
Dataset revision: 06385584fad212b26134c656fdd3ccf9f093f33e

## Pilot subset
Audio recordings: {len(df)}
Unique speakers: {df["speaker"].nunique()}
Recordings per speaker: 20

## Acoustic extraction
Configuration: {config["configuration_name"]}
Features per recording: {len(config["features"])}

Methods:
- Silero VAD + Librosa energy segmentation
- librosa.pyin voiced pitch estimation
- RMS energy statistics

## Quality control
Recordings with internal pauses >=200ms:
{int((df["pause_count_200ms"] > 0).sum())}

Missing pitch: {len(missing_ids)}
Missing audio IDs: {", ".join(missing_ids)}

Original missing values retained.
No imputation performed.

## Descriptive statistics
Mean duration: {df["duration_sec"].mean():.4f} sec
Mean speech ratio: {df["speech_ratio"].mean():.4f}
Mean pause count >=200ms: {df["pause_count_200ms"].mean():.4f}
Mean pause duration: {df["pause_duration_200ms"].mean():.4f} sec
Mean pitch: {df["pitch_mean_hz"].mean():.4f} Hz

## Listening quality assurance
20 preliminary recordings reviewed.
Additional targeted long-pause audits completed.
No expert timestamped pause ground truth.

## Methodological limitations
- Pause detection is not equivalent to hesitation detection.
- Audio leading/trailing silence may reflect recording conditions.
- Pitch estimates may include octave errors.
- Some pYIN recordings have missing pitch values.
- Prompt text is not a verbatim human transcript.
- Pilot QA does not establish general segmentation accuracy.

## Data integrity
Unique audio IDs: PASS
Speaker counts: PASS
Metadata identity: PASS
Feature consistency: PASS
Merged validation: PASS

## Training and test data
No ML training has been conducted in Step 2.
Official test set remains reserved.

## Next research stage
STEP 3: ASR hypothesis generation and transcription
variability analysis (subject to research plan confirmation).
"""

REPORT.write_text(report, encoding="utf-8")

# Mark STEP 2 complete
completed = progress.get("completed_steps", [])
completed = sorted(set(int(x) for x in completed) | {0, 1, 2})

progress["completed_steps"] = completed
progress["current_step"] = 3
progress["step2_status"] = "completed"
progress["step2_pilot_processed_audio"] = 400
progress["step2_final_report"] = str(REPORT)
progress["step2_final_feature_file"] = str(FEATURES)
progress["next_task"] = (
    "Review STEP 3 research design before running ASR"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Backup all small research files, excluding raw audio/parquet
included = []

with zipfile.ZipFile(
    ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as archive:

    for folder_name in [
        "data/metadata",
        "configs",
        "features/acoustic",
        "reports",
        "checkpoints",
        "results/figures"
    ]:
        folder = PROJECT / folder_name
        if not folder.exists():
            continue

        for file in sorted(folder.rglob("*")):
            if not file.is_file():
                continue

            if file.suffix.lower() in {
                ".parquet", ".wav", ".mp3", ".flac", ".zip"
            }:
                continue

            archive.write(
                file,
                arcname=str(file.relative_to(PROJECT))
            )
            included.append(str(file.relative_to(PROJECT)))

with zipfile.ZipFile(ZIP) as archive:
    assert archive.testzip() is None
    archived = set(archive.namelist())

    required = {
        "features/acoustic/pilot_acoustic_features_400.csv",
        "configs/pilot_acoustic_v1.json",
        "reports/step02_final_report.md",
        "reports/step02_merged_validation.json",
        "checkpoints/research_progress.json",
        "data/metadata/pilot_selection_400.csv"
    }

    assert required.issubset(archived)

sha256 = hashlib.sha256(ZIP.read_bytes()).hexdigest()

print("=== STEP 2 FINAL BACKUP ===")
print("Step 2 status:", progress["step2_status"])
print("Completed steps:", progress["completed_steps"])
print("Pilot audio:", len(df))
print("Speakers:", df["speaker"].nunique())
print("Features:", len(config["features"]))
print("Missing pitch:", len(missing_ids))
print("Archived files:", len(included))
print("ZIP:", ZIP)
print("ZIP size (MB):", round(ZIP.stat().st_size / 1048576, 3))
print("ZIP SHA256:", sha256)
print("ZIP integrity: PASS")
print("Next task:", progress["next_task"])
